In [21]:
import pandas as pd
import networkx as nx
import re
import io
import requests
import time

# --- 1. FILE PATHS ---
PATH_SLIM_CSV = 'D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/comparimotif/slimfinder.csv'
PATH_COMPARIMOTIF = 'D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/comparimotif/slimfinder.csv-slimfinder.csv.compare.tdt'
PATH_UNIQUE_MTB = 'D:/2nd  year/2nd term/Thesis/files from klaas/unique mtb genes.xlsx'
PATH_SHARK = 'D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/shark_capture/shark_results_idr/outputs/sharkcapture_consensus_kmers.txt'
PATH_IDR_COORDS = 'D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/analysis_addedmetadata/overview_idrs_with_coords.xlsx'
PATH_CONSERVED_IDR = 'D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/New folder/ESX-1_GOIs_edited.xlsx'

# --- 2. ESX-1 relevant gene list (Rv NAMES) ---
ESX_LIST = [
    "Rv0757", "Rv0758", "Rv0981", "Rv0982", "Rv3597c", "Rv3614c", 
    "Rv3615c", "Rv3616c", "Rv3849", "Rv3862c", "Rv3863", "Rv3864", 
    "Rv3865", "Rv3866", "Rv3867", "Rv3868", "Rv3869", "Rv3870", 
    "Rv3871", "Rv3872", "Rv3873", "Rv3874", "Rv3875", "Rv3876", 
    "Rv3877", "Rv3878", "Rv3879c", "Rv3880c", "Rv3881c", "Rv3882c", "Rv3883c"
]


In [22]:
def get_uniprot_data(file_path, column_name='gene name'):
    """
    Retrieves protein sequences from Uniprot

    Args: 
        file_path (string): Path of file containing target proteins
        column_name='gene name' (string): column containing Rv names of the proteins
    
    Returns:
        Dataframe of protein names, other uniprot identifiers, and sequences.
    """
    df_input = pd.read_excel(file_path)
    # Clean the gene names (remove whitespace)
    gene_list = df_input[column_name].dropna().unique().tolist()

    results = []
    print(f"Starting retrieval for {len(gene_list)} unique genes...")

    for gene in gene_list:
        gene = str(gene).strip()
        print(f"Processing: {gene}")
        
        # Querying UniProt specifically for M. tuberculosis H37Rv (TaxID: 83332)
        url = f"https://rest.uniprot.org/uniprotkb/search?query={gene}+AND+taxonomy_id:83332&fields=accession,gene_names,sequence"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                if data['results']:
                    entry = data['results'][0]
                    
                    # Extract Data
                    acc = entry['primaryAccession']
                    # Get all gene names/locus tags associated with this entry
                    gene_data = entry.get('genes', [{}])[0]
                    primary_name = gene_data.get('geneName', {}).get('value', 'N/A')
                    
                    # Get the list of all locus tags (Rv numbers)
                    locus_tags = [lt.get('value') for lt in gene_data.get('orderedLocusNames', [])]
                    locus_str = ", ".join(locus_tags) if locus_tags else "N/A"
                    
                    sequence = entry['sequence']['value']
                    
                    results.append({
                        'Input_Gene': gene,
                        'UniProt_ACC': acc,
                        'Primary_Gene_Name': primary_name,
                        'Locus_Tags': locus_str,
                        'Sequence': sequence
                    })
                else:
                    print(f"  No UniProt match for {gene}")
            else:
                print(f"  API Error {response.status_code} for {gene}")
        except Exception as e:
            print(f"  Failed {gene}: {e}")
        
        # Rate limiting sleep
        time.sleep(0.3)

    df_output = pd.DataFrame(results)
    return df_output

In [23]:
def label_priority_proteins(df_slims, priority_list):
    """
    Flags whether a motif hit occurred in a high-priority ESX-1 protein.
    
    Args:
        df_slims (pd.DataFrame): dataframe of motif hits.
        priority_list (list): List of gene names (e.g., RV3881C).
        
    Returns:
        pd.DataFrame: Dataframe with an 'is_ESX_related' boolean column.
    """
    # Ensure matching is case-insensitive and clean
    clean_priority = [str(x).strip().upper() for x in priority_list]
    
    df_slims['is_ESX_related'] = df_slims['Gene'].astype(str).str.strip().str.upper().isin(clean_priority)
    
    return df_slims

In [24]:
def get_clustered_families(slim_path, compari_path, threshold=0.4):
    """
    Clusters SLiMFinder motif outputs into families using NormIC.

    Args:
        slim_path (string): Path of SLiMFinder motif hits
        compari_path (string): Path of Comparamotif .tdt file
        threshold (float): NormIC threshold for connection (default=0.4)

    Returns:
        Tuple (pd.dataframe, list) containing:
        Dataframe containing the motif families, their representative and their members
        List containing all motif patterns identified by SLiMFinder
    """
    # Read and clean SLiMFinder CSV
    with open(slim_path, 'r') as f:
        lines = [line.strip().strip('"').replace('""', '"') for line in f.readlines()]
    master_df = pd.read_csv(io.StringIO("\n".join(lines)))
    all_motifs = master_df['Pattern'].unique().tolist()

    # Build Similarity Network
    df_comp = pd.read_csv(compari_path, sep='\t')
    G = nx.Graph()
    G.add_nodes_from(all_motifs)
    for _, row in df_comp.iterrows():
        if row['NormIC'] >= threshold:
            G.add_edge(row['Motif1'], row['Motif2'])
    
    # Extract Families
    families = list(nx.connected_components(G))
    family_data = []
    for i, nodes in enumerate(families, 1):
        subgraph = G.subgraph(nodes)
        hub = max(dict(subgraph.degree()).items(), key=lambda x: x[1])[0]
        family_data.append({'Family_ID': i, 'Hub': hub, 'Members': list(nodes)})
    
    return pd.DataFrame(family_data), all_motifs


In [25]:
def validate_with_shark_seeds(df_families, shark_seeds_path):
    """
    Checks to see if Shark-capture motifs are present in SLiMFinder families.
    
    Args:
        df_families (string): Path for dataframe containing the motif families, their representative and their members
        shark_seeds_path (string): Path for dataframe containing the SHARK-CAPTURE motifs
    
    Returns:
        Input dataframe edited to show whether a SHARK-CAPTURE motif was found in family
    """
    # Load Shark-capture seeds
    shark_df = pd.read_csv(shark_seeds_path)
    seeds = shark_df['consensus'].tolist()
    
    # Add a column to store which seeds were matched
    df_families['Matched_Shark_Seeds'] = [[] for _ in range(len(df_families))]
    df_families['Is_Shark_Validated'] = False

    for idx, fam_row in df_families.iterrows():
        matched_for_this_family = []
        for seed in seeds:
            for motif in fam_row['Members']:
                # Convert SLiMFinder notation 'x' to Regex '.'
                regex_pattern = str(motif).replace('x', '.')
                if re.fullmatch(regex_pattern, seed):
                    matched_for_this_family.append(seed)
                    break # Found a match for this seed in this family
        
        if matched_for_this_family:
            df_families.at[idx, 'Matched_Shark_Seeds'] = list(set(matched_for_this_family))
            df_families.at[idx, 'Is_Shark_Validated'] = True
            
    return df_families

In [33]:
def map_motifs_to_sequences(df_output, df_idrs, all_motifs):
    """
    Map all motif occurrences back to exact proteins and IDR coordinates.
    
    Args:
        df_output (pd.dataframe): dataframe containing the unique MTB protein sequences
        df_idrs (pd.dataframe): dataframe containing the MTB IDRs (supplied by Klaas)
        all_motifs (List of strings): all motif patterns identified by SLiMFinder **check

    Returns:
        pd.dataframe containing the mapping of SLiMFinder motifs to the MTB proteins and whether it's in an IDR
    """
    final_hits = []
    df_idrs['gene_clean'] = df_idrs['gene_name'].str.strip().str.upper()

    for _, prot_row in df_output.iterrows():
        gene = str(prot_row['Input_Gene']).strip().upper()
        protein = str(prot_row['Primary_Gene_Name'])
        seq = str(prot_row['Sequence'])
        seq_length = len(seq)
        gene_idrs = df_idrs[df_idrs['gene_clean'] == gene]
        
        for pattern in all_motifs:
            try:
                for match in re.finditer(f"(?=({pattern}))", seq):
                    m_start, m_end = match.start() + 1, match.start() + len(match.group(1))
                    is_in_idr, idr_range = False, "N/A"
                    
                    for _, idr_row in gene_idrs.iterrows():
                        if m_start >= idr_row['start'] and m_end <= idr_row['end']:
                            is_in_idr = True
                            idr_range = f"{idr_row['start']}-{idr_row['end']}"
                            break
                    
                    final_hits.append({
                        'Gene': gene, 'Gene_name': protein, 'Motif_Pattern': pattern, 'Sequence_Match': match.group(1),
                        'Motif_Start': m_start, 'Motif_End': m_end, 'Sequence_length': seq_length,
                        'Is_in_IDR': is_in_idr, 'IDR_Coordinates': idr_range
                    })
            except: continue
    return pd.DataFrame(final_hits)



In [27]:
def check_conservation_overlap(df_slims, conserved_path):
    """
     Checks if the aligned motifs are in ESX-1 conserved motifs from IDRs_ESX.xlsx.
     
     Args:
        df_slims (pd.dataframe): Dataframe containing the mapping of SLiMFinder motifs to the MTB proteins and whether it's in an IDR
        conserved_path (string): Path to IDRs_ESX file containing the conserved IDRs in ESX-1 related proteins
    
    Returns:
        Tuple(pd.dataframe, pd.dataframe):
        Adapted df_slims dataframe showing boolean column of IDRs found in conserved ESX-1 related proteins
        Dataframe containing the motifs hits that were found in conserved ESX-1 related proteins
    """
    df_cons = pd.read_excel(conserved_path)
    # Parse range string "40-100"
    def parse_range(s):
        m = re.search(r'(\d+)-(\d+)', str(s))
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)
    
    df_cons[['c_start', 'c_end']] = df_cons['position_IDR'].apply(lambda x: pd.Series(parse_range(x)))
    df_slims['gene_clean'] = df_slims['Gene'].astype(str).str.strip().str.upper()
    df_cons['locus_clean'] = df_cons['gene'].str.strip().str.upper()
    
    df_slims['in_conserved_esx_idr'] = False
    detailed_mapping = []

    for idx, row in df_slims.iterrows():
        gene_cons = df_cons[df_cons['locus_clean'] == row['gene_clean']]
        for _, c_row in gene_cons.iterrows():
            if row['Motif_Start'] >= c_row['c_start'] and row['Motif_End'] <= c_row['c_end']:
                df_slims.at[idx, 'in_conserved_esx_idr'] = True
                detailed_mapping.append({**row.to_dict(), 'Conserved_Range': c_row['position_IDR']})
    
    return df_slims, pd.DataFrame(detailed_mapping)

In [28]:
def map_family_and_export_details(df_slims_only, family_df):
    """
    Maps the aligned motifs to families and create the detailed alignment dataframe.
    
    Args: 
    df_slims_only (pd.dataframe): Dataframe containing the motifs hits that were found in conserved ESX-1 related proteins
    family_df (pd.dataframe): Dataframe containing the motif families, their representative and their members
    
    Returns:
    pd.dataframe adapted from input df_slims_only to contain relevant family information
    """

    # 1. Check which families the motifs belong to
    family_lookup = family_df.explode('Members')

    # 2. Clean the strings to ensure perfect matching
    family_lookup['Members'] = family_lookup['Members'].astype(str).str.strip()
    df_slims_only['Motif_Pattern'] = df_slims_only['Motif_Pattern'].astype(str).str.strip()
    
    # 4. Merge
    df_slims_only = df_slims_only.merge(
        family_lookup[['Family_ID', 'Hub', 'Members']], 
        left_on='Motif_Pattern', 
        right_on='Members', 
        how='left'
    )

    # 5. Clean up the dataframe
    df_slims_only = df_slims_only.drop(columns=['Members'])
    df_slims_only = df_slims_only.rename(columns={'Hub': 'Family_Hub'})

    # 6. Check for unassigned motifs
    unassigned = df_slims_only['Family_ID'].isna().sum()
    if unassigned > 0:
        print(f"Warning: {unassigned} hits could not be mapped to a family.")
        
    return df_slims_only

In [38]:
def create_chimera_motif_range(df, gene_name, outfile, attribute_name="motif_region"):
    """
    Creates a .defattr file that colors every residue within a motif.
    
    Args:
        df (pd.DataFrame): df_slims results.
        gene_name (str): The gene to process (e.g., 'RV3881C').
        outfile (str): The filename for the .defattr file.
        attribute_name (str): The name that will appear in Chimera's menu.
    """
    # 1. Filter for the specific protein (case-insensitive)
    gene_df = df[df['Gene'] == gene_name.upper()].copy()
    
    if gene_df.empty:
        print(f"No motif data found for gene: {gene_name}")
        return

    # 2. Initialize ALL residues in the protein to 0
    seq_length = int(gene_df['Sequence_length'].iloc[0])
    residue_dict = {i: 0 for i in range(1, seq_length + 1)}

    for _, row in gene_df.iterrows():
        start = int(row['Motif_Start'])
        end = int(row['Motif_End'])
        
        # Add every number in the range to our set
        for res_id in range(start, end + 1):
            residue_dict[res_id] = 1

    # 3. Write the .defattr file
    with open(outfile, "w", encoding="utf-8") as out:
        out.write(f"attribute: {attribute_name}\n")
        out.write("match mode: 1-to-1\n")
        out.write("recipient: residues\n")
        
        # Write every residue in the motif ranges as 1
        for res in sorted(residue_dict.keys()):
            out.write(f"\t:{res}\t{residue_dict[res]}\n")
            
    print(f"File '{outfile}' created.")

In [ ]:
# Main processing block
# 1. Load data
df_idrs = pd.read_excel(PATH_IDR_COORDS)

# 2. Process Families including
df_families, unique_patterns = get_clustered_families(PATH_SLIM_CSV, PATH_COMPARIMOTIF)
df_families = validate_with_shark_seeds(df_families, PATH_SHARK)

# 3. Map Motifs to sequences and filter for IDRs
df_output = get_uniprot_data(PATH_UNIQUE_MTB, column_name='gene name')
df_results = map_motifs_to_sequences(df_output, df_idrs, unique_patterns)
df_slims_only = df_results[df_results['Is_in_IDR']].copy()

# 4. Align with Shark-capture motifs and Conserved IDRs
df_slims_only = label_priority_proteins(df_slims_only, ESX_LIST)
df_slims_only, detailed_list = check_conservation_overlap(df_slims_only, PATH_CONSERVED_IDR)

# 5. Map Families and Create Detailed DataFrame
df_slims = map_family_and_export_details(
    df_slims_only, 
    df_families
)
conserved_hits = df_slims[df_slims['in_conserved_esx_idr']]

# 6. Final Export
with pd.ExcelWriter('Thesis_Iteration_Results.xlsx') as writer:
    df_slims.to_excel(writer, sheet_name='Summary_Hits', index=False)
    conserved_hits.to_excel(writer, sheet_name='Detailed_Alignment', index=False)
    df_families.to_excel(writer, sheet_name='Family_Clusters', index=False)

print("Processing complete. Outputs saved in 'Thesis_Iteration_Results_2.xlsx'.")



Starting retrieval for 711 unique genes...
Processing: Rv0341
Processing: Rv1975
Processing: Rv2164c
Processing: Rv2131c
Processing: Rv1109c
Processing: Rv2163c
Processing: Rv2383c
Processing: Rv2329c
Processing: Rv0126
Processing: Rv2585c
Processing: Rv3878
Processing: Rv0160c
Processing: Rv0017c
Processing: Rv0280
Processing: Rv3476c
Processing: Rv3882c
Processing: Rv1937
Processing: Rv0449c
Processing: Rv2700
Processing: Rv2770c
Processing: Rv2725c
Processing: Rv3864
Processing: Rv2124c
Processing: Rv3300c
Processing: Rv1039c
Processing: Rv1285
Processing: Rv2777c
Processing: Rv0726c
Processing: Rv3097c
Processing: Rv0556
Processing: Rv1475c
Processing: Rv2841c
Processing: Rv2931
Processing: Rv1664
Processing: Rv1886c
Processing: Rv1427c
Processing: Rv3696c
Processing: Rv2369c
Processing: Rv2125
Processing: Rv2846c
Processing: Rv3630
Processing: Rv1087
Processing: Rv0695
Processing: Rv1351
Processing: Rv2147c
Processing: Rv3479
Processing: Rv3129
Processing: Rv2030c
Processing: Rv34

In [36]:
df_results = map_motifs_to_sequences(df_output, df_idrs, unique_patterns)
df_slims_only = df_results[df_results['Is_in_IDR']].copy()

# 4. Align with Shark-capture motifs and Conserved IDRs
df_slims_only = label_priority_proteins(df_slims_only, ESX_LIST)
df_slims_only, detailed_list = check_conservation_overlap(df_slims_only, PATH_CONSERVED_IDR)

# 5. Map Families and Create Detailed DataFrame
df_slims = map_family_and_export_details(
    df_slims_only, 
    df_families
)
conserved_hits = df_slims[df_slims['in_conserved_esx_idr']]

# 6. Final Export
with pd.ExcelWriter('Thesis_Iteration_Results2.xlsx') as writer:
    df_slims.to_excel(writer, sheet_name='Summary_Hits', index=False)
    conserved_hits.to_excel(writer, sheet_name='Detailed_Alignment', index=False)
    df_families.to_excel(writer, sheet_name='Family_Clusters', index=False)

print("Processing complete. Outputs saved in 'Thesis_Iteration_Results_2.xlsx'.")

Processing complete. Outputs saved in 'Thesis_Iteration_Results_2.xlsx'.


In [39]:
# Visualization
esx_genes = set(conserved_hits['Gene'])

for gene in esx_genes:
    create_chimera_motif_range(conserved_hits, gene, f'{gene}_motifs.defattr')

File 'RV3881C_motifs.defattr' created.
File 'RV3864_motifs.defattr' created.
File 'RV3873_motifs.defattr' created.
File 'RV3876_motifs.defattr' created.
File 'RV3878_motifs.defattr' created.
File 'RV3616C_motifs.defattr' created.
File 'RV3863_motifs.defattr' created.
File 'RV3879C_motifs.defattr' created.
